# Accélérer pandas sur les ventes immobilières (DVF)

Ce notebook applique les techniques du chapitre « Accélération du calcul » à un vrai jeu de données volumineux : les ventes immobilières de France (DVF), 10,8 millions de ventes de 2014 à 2022. Chaque partie mesure ce qu'elle compare, pour que vous voyiez le gain :

1. Mémoire et chargement : mesurer, lire moins, choisir ses types
2. Boucles et vectorisation : la distance de chaque vente à Nantes, puis sa zone
3. Regrouper vite, filtrer lisiblement : `agg` plutôt qu'`apply`, `query` et `eval`
4. Compiler avec Numba : `@njit`, `prange`, une boucle qui ne se vectorise pas, `engine="numba"`
5. Passer à l'échelle : Modin, Polars et DuckDB sur les neuf années

Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin. Les cellules qui suivent une solution affichent les vérifications et les temps : comparez-y vos résultats. Les exercices marqués « facultatif » ne servent pas à la suite : passez-les si le temps manque.

Le notebook tourne sans GPU, sur les 2 cœurs de Colab ou du Codespace, en quelques minutes. Les temps donnés dans les énoncés sont des ordres de grandeur, mesurés sur 2 cœurs : les vôtres varieront, les rapports entre méthodes beaucoup moins.

## Mise en place

La première cellule télécharge DVF depuis shuuchuu/datasets (635 Mo, un dossier Parquet par année) et installe Modin, que Colab n'a pas. La seconde importe les bibliothèques et définit trois outils de mesure :

- `mesurer(fonction, repetitions)` : le meilleur temps, en secondes, de plusieurs appels de `fonction()`, comme `%timeit`, mais en renvoyant le nombre ;
- `chronometrer(fonction)` : le résultat d'un seul appel, long, et son temps ;
- `tableau_temps(temps)` : un dictionnaire de temps en tableau, du plus lent au plus rapide, avec le gain sur le plus lent.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!pip install -q "modin[dask]"

In [ ]:
import time
import timeit

import duckdb
import numpy as np
import pandas as pd
import polars as pl
from geopy.distance import geodesic
from numba import njit, prange

CHEMIN = "dvf/dvf-linearized-2014-2022.parquet"
ANNEE_2022 = [("annee_mutation", "==", 2022)]


def mesurer(fonction, repetitions=3):
  """Le meilleur temps, en secondes, de `repetitions` appels de `fonction()`."""
  return min(timeit.repeat(fonction, number=1, repeat=repetitions))


def chronometrer(fonction):
  """Le résultat d'un appel de `fonction()`, et son temps en secondes."""
  debut = time.perf_counter()
  resultat = fonction()
  return resultat, time.perf_counter() - debut


def tableau_temps(temps):
  """Les temps (nom → secondes), du plus lent au plus rapide, et leur gain."""
  secondes = pd.Series(temps).sort_values(ascending=False)
  return pd.DataFrame(
    {
      "secondes": secondes.round(4),
      "× plus rapide que le plus lent": (secondes.max() / secondes).round(1),
    }
  )

## 1. Mémoire et chargement

DVF est un jeu Parquet partitionné : un dossier par année (`annee_mutation=2014/`…), que `pd.read_parquet` lit d'un bloc. Les parties 1 à 4 travaillent sur l'année 2022 (508 837 ventes, de janvier à juin), que `filters=` lit seule ; la partie 5, sur les neuf années.

In [ ]:
%time dvf = pd.read_parquet(CHEMIN, filters=ANNEE_2022)
dvf.shape

### 1.1 Mesurer la mémoire (facultatif)

Combien de mémoire occupe `dvf` ? Avec `memory_usage(deep=True)`, affichez la mémoire totale de la table en Mo (10⁶ octets), puis ses cinq colonnes les plus lourdes, avec leur type (`dvf.dtypes`). Comparez au total de `memory_usage()` sans `deep=True` : d'où vient l'écart ?

Vous devez trouver environ 206 Mo, dont 155 pour quatre colonnes de chaînes et l'index `id_mutation`, contre 72 Mo sans `deep=True`.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `dvf.memory_usage(deep=True)` renvoie une `Series` : le nombre d'octets de chaque colonne, plus une ligne `Index` ; sa somme divisée par `1e6` donne des Mo.
- `.nlargest(5)` garde ses cinq plus grandes valeurs ; `dvf.dtypes.reindex(...)` donne le type de ces colonnes (`NaN` pour l'`Index`, qui n'en est pas une).
- Sans `deep=True`, une colonne `object` compte 8 octets par ligne : un pointeur vers chaque chaîne, pas la chaîne elle-même.

Un squelette, où chaque `...` est à compléter :

```python
memoire = dvf.memory_usage(deep=True)
print(f"Avec deep=True : {memoire.sum() / 1e6:.0f} Mo")
sans_deep = ...
print(f"Sans deep=True : {sans_deep / 1e6:.0f} Mo")
lourdes = memoire.nlargest(5)
pd.DataFrame({"Mo": (lourdes / 1e6).round(1), "type": ...})
```

#### Solution

In [ ]:
memoire = dvf.memory_usage(deep=True)
print(f"Avec deep=True : {memoire.sum() / 1e6:.0f} Mo")
sans_deep = dvf.memory_usage().sum()
print(f"Sans deep=True : {sans_deep / 1e6:.0f} Mo")
# Les chaînes en object : un objet Python par valeur, que seul deep=True compte.
lourdes = memoire.nlargest(5)
pd.DataFrame(
  {"Mo": (lourdes / 1e6).round(1), "type": dvf.dtypes.reindex(lourdes.index)}
)

### 1.2 Lire moins

La suite du notebook n'utilise que les douze colonnes de `COLONNES` (cellule ci-dessous). Relisez 2022 avec `columns=COLONNES` en plus de `filters=`, dans une variable `dvf` qui remplace la précédente, et mesurez la lecture avec `%time` : comparez-la à celle de la table entière. Affichez la mémoire de la nouvelle table.

L'index `id_mutation`, des chaînes, est relu même avec `columns=`. Retirez-le avec `reset_index(drop=True)`, qui le remplace par un `RangeIndex` (des entiers calculés à la demande, sans mémoire), et affichez de nouveau la mémoire.

Vous devez trouver une lecture environ deux fois plus rapide, 85 Mo, puis 55 Mo sans l'index.

In [ ]:
COLONNES = [
  "mois_mutation",
  "jour_mutation",
  "nature_mutation",
  "valeur_fonciere",
  "code_postal",
  "nom_commune",
  "code_departement",
  "surface_reelle_bati_total",
  "nombre_maisons",
  "nombre_appartements",
  "latitude",
  "longitude",
]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `%time dvf = ...` mesure une affectation et en garde le résultat.
- `filters=ANNEE_2022` comme plus haut ; `columns=` prend une liste de noms de colonnes.
- `drop=True` jette l'index au lieu d'en faire une colonne.

Un squelette, où chaque `...` est à compléter :

```python
%time dvf = pd.read_parquet(CHEMIN, columns=..., filters=...)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
dvf = ...
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo sans l'index")
```

#### Solution

In [ ]:
%time dvf = pd.read_parquet(CHEMIN, columns=COLONNES, filters=ANNEE_2022)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
dvf = dvf.reset_index(drop=True)
print(f"{dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo sans l'index")

### 1.3 Choisir ses types, colonne par colonne

Plutôt que de convertir les colonnes une à une, écrivez `reduire_types(df)`, qui boucle sur les colonnes de `df` avec `df.items()` (des couples nom, `Series`) et renvoie une copie de `df` aux types réduits :

- une colonne `object` dont moins de 10 % des valeurs sont distinctes passe en `category` ;
- une colonne d'entiers passe au plus petit type qui garde toutes ses valeurs, par `pd.to_numeric(..., downcast=...)` : `"unsigned"` si elle n'a aucune valeur négative, `"integer"` sinon.

Les autres colonnes restent telles quelles : `nature_mutation` et `code_departement` sont déjà des `category` (Parquet les stocke en dictionnaire, et pandas les relit ainsi), et un nombre à virgule ne se réduit pas sans perdre de précision. La cellule qui suit la solution vérifie que `reduire_types(dvf)` garde les mêmes valeurs, et affiche sa mémoire.

Vous devez trouver `nom_commune` en `category`, `code_postal` en `uint32`, `nombre_maisons` et `nombre_appartements` en `uint8`, et moins de 25 Mo en tout : plus de huit fois moins que les 206 Mo de la table du départ.

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.api.types.is_integer_dtype.html

In [ ]:
def reduire_types(df):
  """Une copie de `df` aux types réduits."""
  ...  # Votre code ici

#### Conseils supplémentaires

- `for nom, colonne in df.items():` parcourt les colonnes ; `colonne.nunique()` compte ses valeurs distinctes, à comparer à `0.1 * len(colonne)`.
- `colonne.dtype == "object"` repère les chaînes, `pd.api.types.is_integer_dtype(colonne)` les entiers, signés ou non.
- Modifiez une copie, `reduit = df.copy()`, pour laisser `df` intact pendant que vous bouclez dessus.
- Un exemple à vérifier : `reduire_types(pd.DataFrame({"n": range(20), "ville": ["Lyon"] * 20})).dtypes` donne `uint8` pour `n` et `category` pour `ville`.

Un squelette, où chaque `...` est à compléter :

```python
def reduire_types(df):
  reduit = df.copy()
  for nom, colonne in df.items():
    if colonne.dtype == "object" and ...:
      reduit[nom] = ...
    elif pd.api.types.is_integer_dtype(colonne):
      downcast = "unsigned" if ... else "integer"
      reduit[nom] = pd.to_numeric(colonne, downcast=downcast)
  return reduit
```

#### Solution

In [ ]:
def reduire_types(df):
  """Une copie de `df` aux types réduits."""
  reduit = df.copy()
  for nom, colonne in df.items():
    if colonne.dtype == "object" and colonne.nunique() < 0.1 * len(colonne):
      reduit[nom] = colonne.astype("category")
    elif pd.api.types.is_integer_dtype(colonne):
      downcast = "unsigned" if colonne.min() >= 0 else "integer"
      reduit[nom] = pd.to_numeric(colonne, downcast=downcast)
  return reduit

In [ ]:
print(f"Avant : {dvf.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
reduite = reduire_types(dvf)
print(f"Après : {reduite.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
print("Mêmes valeurs :", reduite.astype(object).equals(dvf.astype(object)))

La table telle que la suite l'utilise, aux types réduits par `reduire_types`, et sa mémoire :

In [ ]:
dvf = reduire_types(dvf)
dvf.info(memory_usage="deep")

## 2. Boucles et vectorisation

Quelle distance sépare chaque vente du centre de Nantes ? `geopy.distance.geodesic` la calcule exactement, sur l'ellipsoïde terrestre, mais un point à la fois, en Python : de l'ordre d'un dixième de milliseconde par point. La cellule suivante s'en sert comme référence sur 2 000 ventes de Loire-Atlantique.

In [ ]:
%%time
NANTES = 47.218371, -1.553621  # latitude, longitude du centre de Nantes
loire = dvf[dvf["code_departement"] == "44"]  # Loire-Atlantique
echantillon = loire.head(2000)
reference = np.array(
  [
    geodesic(NANTES, (lat, lon)).km
    for lat, lon in zip(echantillon["latitude"], echantillon["longitude"])
  ]
)

### 2.1 La formule de haversine en NumPy

Sur une sphère de rayon $R$ = 6 371 km, la formule de haversine donne la distance entre deux points de latitudes $\varphi_1$, $\varphi_2$ et de longitudes $\lambda_1$, $\lambda_2$, en radians :

$$d = 2R \arcsin \sqrt{\sin^2 \frac{\varphi_2 - \varphi_1}{2} + \cos \varphi_1 \cos \varphi_2 \sin^2 \frac{\lambda_2 - \lambda_1}{2}}$$

Écrivez `haversine(lat1, lon1, lat2, lon2)`, qui prend des degrés et renvoie des km, avec les seules fonctions de NumPy (`np.radians`, `np.sin`, `np.cos`, `np.arcsin`, `np.sqrt`) : elle marche alors aussi bien sur deux points que sur des colonnes entières, sans boucle. La cellule qui suit la solution la compare à `geodesic` sur les 2 000 ventes d'`echantillon`.

Vous devez trouver `haversine(48.8566, 2.3522, *NANTES)` ≈ 343 km (Paris–Nantes), et un écart relatif à `geodesic` inférieur à 0,5 % : la Terre n'est pas tout à fait une sphère.

**Documentation utile**

- https://numpy.org/doc/stable/reference/generated/numpy.arcsin.html

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """La distance en km entre (lat1, lon1) et (lat2, lon2), donnés en degrés."""
  ...  # Votre code ici

#### Conseils supplémentaires

- Convertissez d'abord les quatre angles en radians, un `np.radians` par variable.
- $\sin^2 x$ s'écrit `np.sin(x) ** 2`.
- Sans `if` ni boucle, la même fonction accepte des nombres, des tableaux NumPy et des `Series` : les opérations de NumPy s'appliquent élément par élément, avec le broadcasting entre un point et une colonne.

Un squelette, où chaque `...` est à compléter :

```python
def haversine(lat1, lon1, lat2, lon2):
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = ...
  a = np.sin((lat2 - lat1) / 2) ** 2 + ...
  return 2 * 6371 * ...
```

#### Solution

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
  """La distance en km entre (lat1, lon1) et (lat2, lon2), donnés en degrés."""
  lat1, lon1 = np.radians(lat1), np.radians(lon1)
  lat2, lon2 = np.radians(lat2), np.radians(lon2)
  a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
  )
  return 2 * 6371 * np.arcsin(np.sqrt(a))

In [ ]:
print(f"Paris–Nantes : {haversine(48.8566, 2.3522, *NANTES):.0f} km")
distances = haversine(*NANTES, echantillon["latitude"], echantillon["longitude"])
ecart = np.abs(distances - reference) / reference
print(f"Écart relatif maximal à geodesic : {ecart.max():.2%}")

### 2.2 Cinq boucles, et une condition sans boucle

La cellule ci-dessous calcule la distance à Nantes de chaque vente de `loire` (18 260 ventes) en appelant `haversine` sur une vente à la fois, de cinq façons (`iterrows`, `apply(axis=1)`, `itertuples`, `zip`, `np.vectorize`), puis d'un seul appel sur les colonnes entières, et affiche leurs temps.

In [ ]:
temps = {}
temps["iterrows"] = mesurer(
  lambda: [
    haversine(*NANTES, vente["latitude"], vente["longitude"])
    for _, vente in loire.iterrows()
  ],
  repetitions=1,
)
temps["apply(axis=1)"] = mesurer(
  lambda: loire.apply(
    lambda vente: haversine(*NANTES, vente["latitude"], vente["longitude"]), axis=1
  ),
  repetitions=1,
)
temps["itertuples"] = mesurer(
  lambda: [
    haversine(*NANTES, vente.latitude, vente.longitude) for vente in loire.itertuples()
  ],
  repetitions=1,
)
temps["zip"] = mesurer(
  lambda: [
    haversine(*NANTES, lat, lon)
    for lat, lon in zip(loire["latitude"], loire["longitude"])
  ],
  repetitions=1,
)
temps["np.vectorize"] = mesurer(
  lambda: np.vectorize(haversine)(*NANTES, loire["latitude"], loire["longitude"]),
  repetitions=1,
)
temps["vectorisée"] = mesurer(
  lambda: haversine(*NANTES, loire["latitude"], loire["longitude"])
)
tableau_temps(temps)

La version vectorisée est des dizaines à des centaines de fois plus rapide que la meilleure boucle. Les cinq boucles appellent `haversine` une fois par vente, en Python, avec des nombres : chaque appel coûte autant que le calcul vectorisé sur des milliers de ventes. `iterrows` et `apply(axis=1)` construisent en plus une `Series` par ligne, et `np.vectorize` n'est qu'une boucle Python plus commode.

Une condition se vectorise aussi. Classez chaque vente de 2022 selon sa distance à Nantes : « Nantes » à moins de 10 km, « agglomération » à moins de 30 km, « reste » au-delà. Calculez d'abord `distances`, la distance à Nantes de toutes les ventes de `dvf`, d'un seul appel de `haversine`. Puis classez-les de deux façons, et rangez leur temps dans le dictionnaire `temps_zones`, sous les noms `"apply"` et `"np.select"` :

- une fonction `zone(distance)`, à `if`, appliquée à chaque distance par `distances.apply(zone)` (une seule répétition) ;
- `np.select`, sans boucle Python.

La cellule qui suit la solution vérifie que les deux donnent les mêmes zones, puis affiche le nombre de ventes de chaque zone et les temps.

Vous devez trouver environ 6 350 ventes dans la zone « Nantes » et 4 270 dans l'« agglomération », et `np.select` une vingtaine de fois plus rapide que `apply`, ou plus.

In [ ]:
temps_zones = {}
# Votre code ici

#### Conseils supplémentaires

- `zone` reçoit un nombre et renvoie une chaîne : `zone(5)` vaut `"Nantes"`, `zone(25)` vaut `"agglomération"` et `zone(100)` vaut `"reste"`.
- `np.select(conditions, choix, default=...)` prend une liste de masques et une liste de valeurs de même longueur : chaque élément reçoit la valeur de la **première** condition vraie, sinon `default`. Après `distances < 10`, la condition `distances < 30` ne retient donc que les ventes entre 10 et 30 km.
- `np.select` se mesure dans une `lambda`, comme `apply` : `mesurer(lambda: np.select(...))`.

Un squelette, où chaque `...` est à compléter :

```python
distances = haversine(*NANTES, dvf["latitude"], dvf["longitude"])


def zone(distance):
  if distance < 10:
    return "Nantes"
  ...


temps_zones = {}
temps_zones["apply"] = mesurer(lambda: distances.apply(zone), repetitions=1)
temps_zones["np.select"] = mesurer(
  lambda: np.select([distances < 10, ...], [...], default=...)
)
```

#### Solution

In [ ]:
distances = haversine(*NANTES, dvf["latitude"], dvf["longitude"])


def zone(distance):
  if distance < 10:
    return "Nantes"
  if distance < 30:
    return "agglomération"
  return "reste"


temps_zones = {}
temps_zones["apply"] = mesurer(lambda: distances.apply(zone), repetitions=1)
temps_zones["np.select"] = mesurer(
  lambda: np.select(
    [distances < 10, distances < 30], ["Nantes", "agglomération"], default="reste"
  )
)

In [ ]:
zones = np.select(
  [distances < 10, distances < 30], ["Nantes", "agglomération"], default="reste"
)
print("Mêmes zones :", (distances.apply(zone).to_numpy() == zones).all())
print(pd.Series(zones).value_counts())
tableau_temps(temps_zones)

## 3. Regrouper vite, filtrer lisiblement

Le prix au m² d'une commune : la somme des valeurs foncières de ses ventes de bâti, divisée par la somme de leurs surfaces. Une commune se repère par son département et son nom, car des communes de départements différents portent le même. La version naïve passe chaque groupe à une fonction Python avec `apply` ; elle prend plusieurs secondes :

In [ ]:
bati = dvf[(dvf["valeur_fonciere"] > 0) & (dvf["surface_reelle_bati_total"] > 0)]
communes = bati.groupby(["code_departement", "nom_commune"], observed=True)


def prix_m2_commune(ventes):
  return ventes["valeur_fonciere"].sum() / ventes["surface_reelle_bati_total"].sum()


%time naif = communes.apply(prix_m2_commune, include_groups=False)
naif.head()

### 3.1 Agréger, puis diviser

Calculez le même prix au m² sans `apply`. Une agrégation nommée de `communes` (`agg`) donne en une fois, par commune, la somme des valeurs (colonne `valeur`), celle des surfaces (`surface`) et le nombre de ventes (`ventes`), dans un DataFrame `sommes` ; la division de deux de ses colonnes donne le prix, `prix_m2`. Mesurez la cellule avec `%%time`, sur sa première ligne. La cellule qui suit la solution vérifie que `prix_m2` est égal à `naif`, et affiche les communes les plus chères parmi celles d'au moins 100 ventes.

Vous devez trouver un calcul des dizaines à des centaines de fois plus rapide, les mêmes prix, et des arrondissements de Paris en tête.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une agrégation nommée prend un couple (colonne, fonction) par colonne du résultat : `agg(valeur=("valeur_fonciere", "sum"), ...)`.
- `"size"` compte les lignes de chaque groupe.
- `"sum"` est une fonction intégrée, en C : un groupe n'est jamais confié à Python.

Un squelette, où chaque `...` est à compléter :

```python
%%time
sommes = communes.agg(
  valeur=("valeur_fonciere", "sum"),
  surface=...,
  ventes=...,
)
prix_m2 = ...
```

#### Solution

In [ ]:
%%time
sommes = communes.agg(
  valeur=("valeur_fonciere", "sum"),
  surface=("surface_reelle_bati_total", "sum"),
  ventes=("valeur_fonciere", "size"),
)
prix_m2 = sommes["valeur"] / sommes["surface"]

In [ ]:
print("Mêmes prix que la version naïve :", np.allclose(prix_m2, naif))
prix_m2[sommes["ventes"] >= 100].nlargest(10).round()

### 3.2 `query` et `eval` (facultatif)

Les ventes d'appartements seuls (sans maison) se filtrent par un masque, dans la cellule ci-dessous, qui mesure aussi ce filtre et le prix au m² de ces ventes.

Écrivez le même filtre avec `dvf.query(...)`, dans une fonction `appartements_query()`, et le prix au m² d'`appartements` avec `appartements.eval(...)`, puis mesurez-les avec `%timeit` : comparez aux deux mesures de la cellule ci-dessous. Affichez le nombre de ventes que garde chaque filtre, et le prix au m² médian d'un appartement.

Vous devez trouver 159 376 ventes par les deux filtres, `query` environ deux fois plus lent que le masque et `eval` une dizaine de fois plus lent que la division : leur intérêt est la lisibilité, pas la vitesse.

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.eval.html

In [ ]:
def appartements_masque():
  return dvf[
    (dvf["nombre_appartements"] > 0)
    & (dvf["nombre_maisons"] == 0)
    & (dvf["valeur_fonciere"] > 0)
    & (dvf["surface_reelle_bati_total"] > 0)
  ]


appartements = appartements_masque()
%timeit appartements_masque()
%timeit appartements["valeur_fonciere"] / appartements["surface_reelle_bati_total"]

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Dans le texte d'une requête, les colonnes se nomment sans `dvf[...]`, et `and` remplace `&` ; deux chaînes côte à côte, sur deux lignes, n'en font qu'une.
- `eval` prend une expression sur les colonnes, et renvoie une `Series`.

Un squelette, où chaque `...` est à compléter :

```python
def appartements_query():
  return dvf.query("nombre_appartements > 0 and nombre_maisons == 0 and ...")


%timeit appartements_query()
%timeit appartements.eval(...)
print(len(appartements_query()), len(appartements), "ventes")
prix = appartements.eval(...)
print(f"Prix médian d'un appartement : {prix.median():.0f} €/m²")
```

#### Solution

In [ ]:
def appartements_query():
  return dvf.query(
    "nombre_appartements > 0 and nombre_maisons == 0"
    " and valeur_fonciere > 0 and surface_reelle_bati_total > 0"
  )


%timeit appartements_query()
%timeit appartements.eval("valeur_fonciere / surface_reelle_bati_total")
print(len(appartements_query()), len(appartements), "ventes")
prix = appartements.eval("valeur_fonciere / surface_reelle_bati_total")
print(f"Prix médian d'un appartement : {prix.median():.0f} €/m²")

## 4. Compiler avec Numba

À quelle distance de la plus proche des dix plus grandes villes de France se trouve chaque vente ? Il faut, pour chacune des 508 837 ventes, ses dix distances aux villes, puis la plus petite. En NumPy, le broadcasting d'une colonne `(n, 1)` contre une ligne `(10,)` calcule les dix distances de chaque vente d'un coup, dans un tableau temporaire `(n, 10)` : c'est la référence, mesurée ci-dessous.

`haversine_njit` est votre `haversine` compilée par Numba (`njit(haversine)`), pour être appelée depuis une autre fonction Numba.

In [ ]:
VILLES = pd.DataFrame(
  [
    ("Paris", 48.8566, 2.3522),
    ("Marseille", 43.2965, 5.3698),
    ("Lyon", 45.7640, 4.8357),
    ("Toulouse", 43.6047, 1.4442),
    ("Nice", 43.7102, 7.2620),
    ("Nantes", 47.2184, -1.5536),
    ("Montpellier", 43.6108, 3.8767),
    ("Strasbourg", 48.5734, 7.7521),
    ("Bordeaux", 44.8378, -0.5792),
    ("Lille", 50.6292, 3.0573),
  ],
  columns=["ville", "latitude", "longitude"],
)
lat, lon = dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy()
lat_v, lon_v = VILLES["latitude"].to_numpy(), VILLES["longitude"].to_numpy()
haversine_njit = njit(haversine)


def distance_ville_proche_numpy(lat, lon, lat_v, lon_v):
  distances = haversine(lat[:, np.newaxis], lon[:, np.newaxis], lat_v, lon_v)
  return distances.min(axis=1)  # la plus petite des 10 distances de chaque vente


temps_numba = {
  "NumPy": mesurer(lambda: distance_ville_proche_numpy(lat, lon, lat_v, lon_v))
}
temps_numba

### 4.1 `@njit`, puis `parallel=True`

Numba compile une fonction Python qui boucle sur des tableaux NumPy. Écrivez `distance_ville_proche(lat, lon, lat_v, lon_v)`, décorée par `@njit`, avec deux boucles : pour chaque vente `i`, la plus petite des distances `haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j])` aux villes `j`, sans tableau temporaire.

Puis écrivez `distance_ville_proche_parallele`, la même fonction décorée par `@njit(parallel=True)`, dont la boucle sur les ventes passe de `range` à `prange` : ses itérations, indépendantes, se répartissent sur les cœurs.

Mesurez le premier appel de chacune avec `%time` : il compile la fonction. Puis ajoutez leur temps, `mesurer(lambda: ...)`, à `temps_numba`, sous les noms `"@njit"` et `"@njit(parallel=True)"`. La cellule qui suit la solution vérifie leurs résultats et affiche les temps.

Vous devez trouver une compilation de l'ordre d'une seconde, `@njit` un peu plus rapide que NumPy, qui remplit puis parcourt son tableau temporaire, et `parallel=True` jusqu'à deux fois plus rapide encore sur 2 cœurs.

In [ ]:
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v): ...  # Votre code ici


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v): ...  # Votre code ici

#### Conseils supplémentaires

- `np.empty(len(lat))` réserve le tableau du résultat ; `np.inf` est un bon point de départ pour une plus petite valeur, que `min(a, b)` met à jour.
- Dans une fonction `@njit`, une boucle sur des nombres est compilée, aussi rapide qu'en C : c'est ce qui permet de se passer du tableau `(n, 10)`.
- `prange` vient de `numba`, importé au début du notebook ; seule la boucle extérieure devient `prange`.
- Une `TypingError` de Numba vient souvent d'un argument qui n'est pas un tableau NumPy (une `Series`, par exemple) : passez `lat`, `lon`, `lat_v` et `lon_v`.
- `%time proche = ...` : sans affectation, `%time` afficherait tout le tableau.

Un squelette, où chaque `...` est à compléter :

```python
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in range(len(lat)):
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, ...)
    proche[i] = plus_petite
  return proche


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v): ...


%time proche = distance_ville_proche(lat, lon, lat_v, lon_v)
%time proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
temps_numba["@njit"] = mesurer(lambda: distance_ville_proche(lat, lon, lat_v, lon_v))
temps_numba["@njit(parallel=True)"] = ...
```

#### Solution

In [ ]:
@njit
def distance_ville_proche(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in range(len(lat)):
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j]))
    proche[i] = plus_petite
  return proche


@njit(parallel=True)
def distance_ville_proche_parallele(lat, lon, lat_v, lon_v):
  proche = np.empty(len(lat))
  for i in prange(len(lat)):  # itérations indépendantes, réparties sur les cœurs
    plus_petite = np.inf
    for j in range(len(lat_v)):
      plus_petite = min(plus_petite, haversine_njit(lat[i], lon[i], lat_v[j], lon_v[j]))
    proche[i] = plus_petite
  return proche


%time proche = distance_ville_proche(lat, lon, lat_v, lon_v)
%time proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
temps_numba["@njit"] = mesurer(lambda: distance_ville_proche(lat, lon, lat_v, lon_v))
temps_numba["@njit(parallel=True)"] = mesurer(
  lambda: distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
)

In [ ]:
proche = distance_ville_proche_parallele(lat, lon, lat_v, lon_v)
reference_numpy = distance_ville_proche_numpy(lat, lon, lat_v, lon_v)
print("Mêmes distances qu'en NumPy :", np.allclose(proche, reference_numpy))
print(f"{(proche < 20).mean():.0%} des ventes à moins de 20 km d'une des dix villes")
tableau_temps(temps_numba)

### 4.2 Une boucle qui ne se vectorise pas : un prix de référence

Numba sert surtout aux boucles dont chaque pas dépend du précédent, qu'aucune opération sur des colonnes entières ne remplace. La cellule ci-dessous donne le prix au m² des ventes parisiennes de 2022, dans l'ordre de leurs dates (`prix_paris`), et sa tendance, la médiane glissante sur 500 ventes, en tableau NumPy (`tendance`).

Un prix de référence suit cette tendance, mais ne change que quand elle s'en écarte de plus d'un seuil : il part de la première valeur, et prend la valeur `prix[i]` quand l'écart entre `prix[i]` et la référence dépasse `seuil` fois la référence. Écrivez `prix_reference(prix, seuil)`, une fonction Python sans décorateur, qui renvoie le tableau des références, de la longueur de `prix`. Compilez-la avec `prix_reference_njit = njit(prix_reference)`, comme `haversine_njit`, et mesurez avec `%time` son premier appel sur `tendance`, avec un seuil de 2 %. Puis rangez les temps des deux versions dans `temps_reference`, sous les noms `"Python"` et `"@njit"`. La cellule qui suit la solution vérifie qu'elles donnent les mêmes références, et compte les changements.

Vous devez trouver une référence qui ne change qu'une vingtaine de fois sur près de 17 500 ventes, et la version compilée environ cent fois plus rapide que Python.

In [ ]:
paris = bati[bati["code_departement"] == "75"].sort_values(
  ["mois_mutation", "jour_mutation"], ignore_index=True
)
prix_paris = paris["valeur_fonciere"] / paris["surface_reelle_bati_total"]
tendance = prix_paris.rolling(500).median().dropna().to_numpy()
len(prix_paris), len(tendance)

In [ ]:
def prix_reference(prix, seuil):
  """La référence qui suit `prix` quand il s'en écarte de plus de `seuil`."""
  ...  # Votre code ici

#### Conseils supplémentaires

- Une boucle sur `range(len(prix))` garde la référence courante dans une variable, que met à jour un `if`, et l'écrit à chaque pas dans un tableau réservé par `np.empty(len(prix))`.
- `abs(a - b) > seuil * b` compare un écart relatif sans division.
- Un exemple à vérifier : `prix_reference(np.array([10000.0, 10100, 10300, 10250, 9900]), 0.02)` donne `[10000, 10000, 10300, 10300, 9900]` : 10 100 est à 1 % de 10 000, 10 300 à 3 %, 10 250 à moins de 1 % de 10 300, et 9 900 à près de 4 %.
- Numba compile pour des tableaux NumPy : `tendance` en est un, `prix_paris` (une `Series`) non.
- `%time reference = ...` : sans affectation, `%time` afficherait tout le tableau.

Un squelette, où chaque `...` est à compléter :

```python
def prix_reference(prix, seuil):
  reference = np.empty(len(prix))
  courante = prix[0]
  for i in range(len(prix)):
    if ...:
      courante = prix[i]
    reference[i] = ...
  return reference


prix_reference_njit = njit(prix_reference)
%time reference = prix_reference_njit(tendance, 0.02)
temps_reference = {}
temps_reference["Python"] = mesurer(lambda: prix_reference(tendance, 0.02))
temps_reference["@njit"] = ...
```

#### Solution

In [ ]:
def prix_reference(prix, seuil):
  """La référence qui suit `prix` quand il s'en écarte de plus de `seuil`."""
  reference = np.empty(len(prix))
  courante = prix[0]
  for i in range(len(prix)):
    if abs(prix[i] - courante) > seuil * courante:
      courante = prix[i]  # chaque pas dépend du précédent
    reference[i] = courante
  return reference


prix_reference_njit = njit(prix_reference)
%time reference = prix_reference_njit(tendance, 0.02)
temps_reference = {}
temps_reference["Python"] = mesurer(lambda: prix_reference(tendance, 0.02))
temps_reference["@njit"] = mesurer(lambda: prix_reference_njit(tendance, 0.02))

In [ ]:
reference = prix_reference_njit(tendance, 0.02)
egal = np.array_equal(reference, prix_reference(tendance, 0.02))
print("Mêmes références avec et sans @njit :", egal)
print(f"{(np.diff(reference) != 0).sum()} changements sur {len(reference)} ventes")
tableau_temps(temps_reference)

### 4.3 `engine="numba"` dans pandas (facultatif)

Avec `engine="numba"`, pandas compile lui-même une fonction par Numba : dans `rolling(...).apply(f, raw=True)`, qui appelle `f` sur chaque fenêtre, un tableau NumPy, et dans l'`agg` d'un `groupby`. Mesurez-le dans ces deux cas, contre la méthode intégrée qui fait le même calcul :

- la médiane glissante de `prix_paris` sur 500 ventes : `rolling(500).apply(np.median, raw=True, engine="numba")` contre `rolling(500).median()`, dans `temps_glissant` ;
- l'écart interquartile (IQR) du prix au m² de chaque commune, `np.percentile(x, 75) - np.percentile(x, 25)` sur les prix de ses ventes (`prix_communes`, cellule ci-dessous) : `agg(iqr_groupe, engine="numba")`, où `iqr_groupe(values, index)` a la signature qu'impose pandas, contre `quantile(0.75) - quantile(0.25)`, dans `temps_iqr`.

Rangez chaque temps sous le nom `"Numba"` ou sous celui de la méthode intégrée, et mesurez avec `%time` le premier appel de chaque version Numba : il compile la fonction. La cellule qui suit la solution vérifie les résultats et affiche les temps.

Vous devez trouver `median()` une dizaine de fois plus rapide que Numba, mais Numba plus rapide que les deux `quantile`, qui trient chaque groupe deux fois : une méthode intégrée n'est pas toujours la plus rapide, seule la mesure les départage.

In [ ]:
prix_bati = bati["valeur_fonciere"] / bati["surface_reelle_bati_total"]
prix_communes = prix_bati.groupby(
  [bati["code_departement"], bati["nom_commune"]], observed=True
)

In [ ]:
temps_glissant = {}
temps_iqr = {}
# Votre code ici

#### Conseils supplémentaires

- `raw=True` passe chaque fenêtre en tableau NumPy, ce qu'exige `engine="numba"` : Numba ne sait pas compiler une `Series`.
- Avec `engine="numba"`, pandas passe à la fonction d'`agg` les valeurs du groupe et son index, deux tableaux NumPy : `def iqr_groupe(values, index)`, même si elle n'utilise pas `index`.
- `engine="numba"` lève une `TypingError` quand l'index de la `Series` est fait de chaînes : celui de `prix_bati` vient du `RangeIndex` de la partie 1 (`reset_index(drop=True)`).
- Le premier appel compile ; les suivants réutilisent le code compilé, gardé en cache par pandas.

Un squelette, où chaque `...` est à compléter :

```python
fenetres = prix_paris.rolling(500)
%time mediane = fenetres.apply(np.median, raw=True, engine="numba")
temps_glissant = {}
temps_glissant["Numba"] = mesurer(lambda: ...)
temps_glissant["median()"] = mesurer(lambda: fenetres.median())


def iqr_groupe(values, index):
  return ...


%time iqr = prix_communes.agg(iqr_groupe, engine="numba")
temps_iqr = {}
temps_iqr["Numba"] = ...
temps_iqr["quantile"] = mesurer(lambda: ...)
```

#### Solution

In [ ]:
fenetres = prix_paris.rolling(500)
%time mediane = fenetres.apply(np.median, raw=True, engine="numba")
temps_glissant = {}
temps_glissant["Numba"] = mesurer(
  lambda: fenetres.apply(np.median, raw=True, engine="numba")
)
temps_glissant["median()"] = mesurer(lambda: fenetres.median())


def iqr_groupe(values, index):
  return np.percentile(values, 75) - np.percentile(values, 25)


%time iqr = prix_communes.agg(iqr_groupe, engine="numba")
temps_iqr = {}
temps_iqr["Numba"] = mesurer(lambda: prix_communes.agg(iqr_groupe, engine="numba"))
temps_iqr["quantile"] = mesurer(
  lambda: prix_communes.quantile(0.75) - prix_communes.quantile(0.25)
)

In [ ]:
mediane = prix_paris.rolling(500).median()
mediane_numba = prix_paris.rolling(500).apply(np.median, raw=True, engine="numba")
print("Mêmes médianes :", np.allclose(mediane, mediane_numba, equal_nan=True))
iqr = prix_communes.agg(iqr_groupe, engine="numba")
iqr_quantile = prix_communes.quantile(0.75) - prix_communes.quantile(0.25)
print("Mêmes IQR :", np.allclose(iqr, iqr_quantile))
pd.concat(
  {
    "médiane glissante": tableau_temps(temps_glissant),
    "IQR par commune": tableau_temps(temps_iqr),
  }
)

## 5. Passer à l'échelle : 2014–2022

Le prix au m² médian de chaque département, sur les neuf années : 10,8 millions de ventes. pandas lit les trois colonnes utiles (`COLONNES_PRIX`), et l'index `id_mutation` avec elles, puis `prix_medians` filtre et groupe : une dizaine de secondes ou plus, et plus de 2 Go de mémoire au plus fort. `par_departement` met un résultat sous une forme commune (une `Series` indexée par le code du département, en texte, triée), pour comparer ceux de deux bibliothèques.

In [ ]:
COLONNES_PRIX = ["code_departement", "valeur_fonciere", "surface_reelle_bati_total"]


def prix_medians(dvf):
  """Le prix au m² médian du bâti de chaque département de `dvf`."""
  bati = dvf.query("surface_reelle_bati_total > 0")
  prix = bati["valeur_fonciere"] / bati["surface_reelle_bati_total"]
  return prix.groupby(bati["code_departement"], observed=True).median()


def prix_departements_pandas():
  return prix_medians(pd.read_parquet(CHEMIN, columns=COLONNES_PRIX))


def par_departement(prix):
  """`prix` en Series indexée par le code du département, en texte, triée."""
  return pd.Series(prix.to_numpy(), index=prix.index.astype(str)).sort_index()


temps_echelle = {}
prix_pandas, temps_echelle["pandas"] = chronometrer(prix_departements_pandas)
print(f"pandas : {temps_echelle['pandas']:.1f} s")
prix_pandas.nlargest(5).round()

### 5.1 Où Modin perd-il son temps ?

Modin reprend l'API de pandas sur plusieurs cœurs : il suffit d'importer `modin.pandas` à la place de `pandas`. La cellule ci-dessous fait le même calcul avec Modin : la même fonction `prix_medians`, sur une table lue par `mpd.read_parquet`, et un résultat ramené en pandas par `.modin.to_pandas()`. Modin démarre d'abord ses processus (moteur Dask), et prévient qu'il lit ce Parquet partitionné avec pandas (« Defaulting to pandas ») avant de répartir la table : c'est normal.

In [ ]:
import modin.pandas as mpd


def prix_departements_modin():
  dvf = mpd.read_parquet(CHEMIN, columns=COLONNES_PRIX)
  return prix_medians(dvf).modin.to_pandas()


prix_modin, temps_echelle["Modin"] = chronometrer(prix_departements_modin)
egal = np.allclose(par_departement(prix_modin), par_departement(prix_pandas))
print("Même résultat que pandas :", egal)
tableau_temps(temps_echelle)

Modin est plus lent que pandas sur 2 cœurs. Pour savoir où il perd son temps, mesurez à part, avec `chronometrer`, les trois étapes de `prix_departements_modin` : la lecture (`mpd.read_parquet`), le calcul (`prix_medians`) et la conversion (`.modin.to_pandas()`), puis les deux premières en pandas. Rangez les temps dans `etapes`, un dictionnaire par bibliothèque, sous les noms `"lecture"`, `"calcul"` et `"conversion"` ; la cellule qui suit la solution les affiche en tableau.

Vous devez trouver une lecture plus de deux fois plus lente avec Modin, qui lit le Parquet avec pandas, puis envoie la table à ses processus : c'est là qu'il perd son temps. Le calcul, réparti sur les 2 cœurs, ne rattrape pas ce retard, et la conversion est négligeable, le résultat n'ayant qu'une centaine de lignes.

**Documentation utile**

- https://modin.readthedocs.io/en/stable/flow/modin/experimental/pandas.html#modin.pandas.DataFrame.modin.to_pandas

In [ ]:
etapes = {"pandas": {}, "Modin": {}}
# Votre code ici

#### Conseils supplémentaires

- `chronometrer` prend une fonction sans argument : `chronometrer(lambda: pd.read_parquet(CHEMIN, columns=COLONNES_PRIX))` renvoie la table et son temps.
- Gardez le résultat de chaque étape pour la suivante : la table lue pour `prix_medians`, ses médianes pour `.modin.to_pandas()`.
- La table de pandas occupe près d'un gigaoctet : `del dvf_pandas`, une fois son calcul mesuré, la libère avant la lecture de Modin.

Un squelette, où chaque `...` est à compléter :

```python
etapes = {"pandas": {}, "Modin": {}}
dvf_pandas, etapes["pandas"]["lecture"] = chronometrer(
  lambda: pd.read_parquet(CHEMIN, columns=COLONNES_PRIX)
)
_, etapes["pandas"]["calcul"] = chronometrer(lambda: prix_medians(dvf_pandas))
del dvf_pandas
dvf_modin, etapes["Modin"]["lecture"] = ...
medianes, etapes["Modin"]["calcul"] = ...
_, etapes["Modin"]["conversion"] = ...
```

#### Solution

In [ ]:
etapes = {"pandas": {}, "Modin": {}}
dvf_pandas, etapes["pandas"]["lecture"] = chronometrer(
  lambda: pd.read_parquet(CHEMIN, columns=COLONNES_PRIX)
)
_, etapes["pandas"]["calcul"] = chronometrer(lambda: prix_medians(dvf_pandas))
del dvf_pandas  # libère sa mémoire avant la lecture de Modin
dvf_modin, etapes["Modin"]["lecture"] = chronometrer(
  lambda: mpd.read_parquet(CHEMIN, columns=COLONNES_PRIX)
)
medianes, etapes["Modin"]["calcul"] = chronometrer(lambda: prix_medians(dvf_modin))
_, etapes["Modin"]["conversion"] = chronometrer(lambda: medianes.modin.to_pandas())
del dvf_modin

In [ ]:
pd.DataFrame(etapes).round(2)

### 5.2 Polars : les appartements parisiens, année par année

Une autre question, sur les neuf années : le prix au m² médian des appartements de Paris (les ventes d'appartements seuls, sans maison, de valeur et de surface positives, comme en 3.2), année par année. La cellule ci-dessous y répond en pandas, en ne lisant que les ventes de Paris (`filters=`), et mesure le calcul dans `temps_paris`.

Écrivez `paris_par_annee_polars()`, le même calcul en Polars : une requête paresseuse sur le dossier Parquet, `pl.scan_parquet(CHEMIN)`, puis `filter`, `group_by` de l'année (`annee_mutation`), `agg` de la médiane du prix au m², nommée `prix_m2` (`alias`), `sort` par année et `collect()`, qui l'exécute. Renvoyez-en une `Series` pandas indexée par l'année (`.to_pandas()`, `set_index`). Mesurez-la avec `chronometrer`, dans `par_annee_polars` et `temps_paris["Polars"]`.

Vous devez trouver Polars plus rapide que pandas, qui ne lit pourtant lui aussi que les ventes de Paris, et un prix médian passé d'environ 8 000 € le m² en 2014 à plus de 10 600 € de 2020 à 2022.

In [ ]:
def paris_par_annee_pandas():
  dvf = pd.read_parquet(
    CHEMIN,
    columns=[
      "annee_mutation",
      "valeur_fonciere",
      "surface_reelle_bati_total",
      "nombre_maisons",
      "nombre_appartements",
    ],
    filters=[("code_departement", "==", "75")],
  )
  appartements = dvf.query(
    "nombre_appartements > 0 and nombre_maisons == 0"
    " and valeur_fonciere > 0 and surface_reelle_bati_total > 0"
  )
  prix = appartements["valeur_fonciere"] / appartements["surface_reelle_bati_total"]
  return prix.groupby(appartements["annee_mutation"], observed=True).median()


temps_paris = {}
par_annee_pandas, temps_paris["pandas"] = chronometrer(paris_par_annee_pandas)
print(f"pandas : {temps_paris['pandas']:.1f} s")
par_annee_pandas.round()

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `pl.scan_parquet(CHEMIN)`, sur le dossier : Polars reconnaît ses sous-dossiers `annee_mutation=2014`… et en fait la colonne `annee_mutation` ; avec le motif `f"{CHEMIN}/**/*.parquet"`, il ne la crée pas.
- `pl.col("nom")` désigne une colonne dans une expression ; les expressions se combinent comme des colonnes pandas (`/`, `>`, `&`, une condition entre parenthèses) et ont leurs méthodes (`.median()`, `.alias("nom")`).
- Rien n'est lu avant `collect()` : Polars optimise d'abord tout le plan.
- Le DataFrame Polars n'a pas d'index : `.to_pandas().set_index("annee_mutation")["prix_m2"]`.

Un squelette, où chaque `...` est à compléter :

```python
def paris_par_annee_polars():
  prix = (
    pl.scan_parquet(CHEMIN)
    .filter(
      (pl.col("code_departement") == "75")
      & (pl.col("nombre_appartements") > 0)
      & (pl.col("nombre_maisons") == 0)
      & ...
    )
    .group_by("annee_mutation")
    .agg(...)
    .sort("annee_mutation")
    .collect()
  )
  return ...


par_annee_polars, temps_paris["Polars"] = chronometrer(paris_par_annee_polars)
print(f"Polars : {temps_paris['Polars']:.1f} s")
```

#### Solution

In [ ]:
def paris_par_annee_polars():
  prix = (
    pl.scan_parquet(CHEMIN)
    .filter(
      (pl.col("code_departement") == "75")
      & (pl.col("nombre_appartements") > 0)
      & (pl.col("nombre_maisons") == 0)
      & (pl.col("valeur_fonciere") > 0)
      & (pl.col("surface_reelle_bati_total") > 0)
    )
    .group_by("annee_mutation")
    .agg(
      (pl.col("valeur_fonciere") / pl.col("surface_reelle_bati_total"))
      .median()
      .alias("prix_m2")
    )
    .sort("annee_mutation")
    .collect()
  )
  return prix.to_pandas().set_index("annee_mutation")["prix_m2"]


par_annee_polars, temps_paris["Polars"] = chronometrer(paris_par_annee_polars)
print(f"Polars : {temps_paris['Polars']:.1f} s")

### 5.3 DuckDB

Écrivez `paris_par_annee_duckdb()`, le même calcul en SQL : `duckdb.sql(...)` sur les fichiers Parquet (`FROM '...'`, avec le motif `**/*.parquet` : DuckDB, lui, reconnaît les dossiers par année et ajoute la colonne `annee_mutation`), `median(...)` du prix au m², `WHERE`, `GROUP BY`, `ORDER BY`, puis `.df()` pour un DataFrame pandas, dont vous renvoyez une `Series` indexée par l'année. Mesurez-la avec `chronometrer`, dans `par_annee_duckdb` et `temps_paris["DuckDB"]`. La cellule qui suit la solution compare les résultats et les temps des trois bibliothèques.

Vous devez trouver les mêmes prix qu'en pandas et en Polars, et DuckDB plus rapide encore que Polars.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Une f-string insère `CHEMIN` dans la requête ; dans la requête, le chemin est entre apostrophes, comme le code du département (`'75'`).
- `AND` combine les conditions du `WHERE` ; `AS prix_m2` nomme la colonne du résultat.

Un squelette, où chaque `...` est à compléter :

```python
def paris_par_annee_duckdb():
  prix = duckdb.sql(f"""
    SELECT annee_mutation,
           median(...) AS prix_m2
    FROM '{CHEMIN}/**/*.parquet'
    WHERE code_departement = '75'
      AND ...
    GROUP BY annee_mutation
    ORDER BY annee_mutation
  """).df()
  return ...


par_annee_duckdb, temps_paris["DuckDB"] = chronometrer(paris_par_annee_duckdb)
print(f"DuckDB : {temps_paris['DuckDB']:.1f} s")
```

#### Solution

In [ ]:
def paris_par_annee_duckdb():
  prix = duckdb.sql(f"""
    SELECT annee_mutation,
           median(valeur_fonciere / surface_reelle_bati_total) AS prix_m2
    FROM '{CHEMIN}/**/*.parquet'
    WHERE code_departement = '75'
      AND nombre_appartements > 0 AND nombre_maisons = 0
      AND valeur_fonciere > 0 AND surface_reelle_bati_total > 0
    GROUP BY annee_mutation
    ORDER BY annee_mutation
  """).df()
  return prix.set_index("annee_mutation")["prix_m2"]


par_annee_duckdb, temps_paris["DuckDB"] = chronometrer(paris_par_annee_duckdb)
print(f"DuckDB : {temps_paris['DuckDB']:.1f} s")

In [ ]:
for nom, par_annee in [("Polars", par_annee_polars), ("DuckDB", par_annee_duckdb)]:
  egal = np.allclose(par_annee.to_numpy(), par_annee_pandas.to_numpy())
  print(f"{nom} donne les mêmes prix que pandas : {egal}")
tableau_temps(temps_paris)

## À retenir

- Mesurer d'abord : `%timeit`, `%time`, `memory_usage(deep=True)`.
- Lire moins (`columns=`, `filters=`) et choisir ses types (`category`, `downcast`), colonne par colonne avec `df.items()` : souvent le premier gain, et le plus simple.
- Vectoriser : une opération sur des colonnes entières plutôt qu'une boucle, `apply` ou `np.vectorize`, qui appellent Python à chaque ligne ; `np.select` plutôt qu'une fonction à `if`.
- Grouper avec des fonctions intégrées (`agg` de `"sum"`, `"size"`…), pas avec `apply` ; `query` et `eval` servent la lisibilité.
- Numba, quand une boucle ne se vectorise pas, ou mal : `@njit`, `prange`, `engine="numba"`, et une mesure contre la méthode intégrée.
- Au-delà de la mémoire, ou pour aller plus vite : Polars et DuckDB plutôt que Modin, sur une machine à peu de cœurs.